# Categorical Encoding

Machine learning models require numerical input. Categorical encoding converts text labels into numbers in a way that preserves the right information for the model.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

df = sns.load_dataset('titanic')
print("Categorical columns:", df.select_dtypes(include=['object', 'category']).columns.tolist())

Categorical columns: ['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive']


## Why Encoding Matters

The choice of encoding method determines what mathematical relationships the model can learn:

| Encoding | Creates | Model sees |
|---|---|---|
| Label | 1 column of integers | A numeric ordering (may be wrong) |
| One-Hot | k binary columns | k independent presence/absence flags |

---
## Label Encoding

Assigns each unique category an integer (0, 1, 2, ...).

**Use when the column is ordinal** — the categories have a meaningful order (small < medium < large).

**Danger with nominal data**: the model will treat the integers as a numeric scale. If `red=0`, `blue=1`, `green=2`, the model treats `green` as twice `blue` — which is meaningless.

In [2]:
# Good use: ordinal column
size_df = pd.DataFrame({'size': ['small', 'large', 'medium', 'small', 'large', 'medium']})
size_order = {'small': 0, 'medium': 1, 'large': 2}
size_df['size_encoded'] = size_df['size'].map(size_order)
print("Ordinal encoding (meaningful order):")
print(size_df)

Ordinal encoding (meaningful order):
     size  size_encoded
0   small             0
1   large             2
2  medium             1
3   small             0
4   large             2
5  medium             1


In [3]:
# sklearn LabelEncoder
le = LabelEncoder()
df_enc = df[['sex', 'embarked']].dropna().copy()

df_enc['sex_label'] = le.fit_transform(df_enc['sex'])
print("Label encoding of 'sex':")
print(df_enc[['sex', 'sex_label']].drop_duplicates().sort_values('sex_label'))
print()
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Label encoding of 'sex':
      sex  sex_label
1  female          0
0    male          1

Class mapping: {'female': np.int64(0), 'male': np.int64(1)}


In [4]:
# Problem: label encoding implies order for nominal data
embarked_le = LabelEncoder()
df_enc['embarked_label'] = embarked_le.fit_transform(df_enc['embarked'])
print("Label encoding of 'embarked' (nominal — order is arbitrary):")
print(df_enc[['embarked', 'embarked_label']].drop_duplicates().sort_values('embarked_label'))
print()
print("Q: Is 'S' (=2) really twice as 'embarked' as 'C' (=0)?  → No. Bad encoding.")

Label encoding of 'embarked' (nominal — order is arbitrary):
  embarked  embarked_label
1        C               0
5        Q               1
0        S               2

Q: Is 'S' (=2) really twice as 'embarked' as 'C' (=0)?  → No. Bad encoding.


---
## One-Hot Encoding (OHE)

Creates one binary column per category. A row has a `1` in the column for its category and `0` everywhere else.

**Use when the column is nominal** — no meaningful ordering between categories.

**Dummy variable trap**: if there are k categories, include only k−1 columns. The omitted category is the baseline — the model can infer it when all k−1 columns are 0.

In [5]:
# pandas get_dummies (drop_first=True avoids dummy trap)
df_ohe = pd.get_dummies(df[['embarked']], drop_first=True)
print("One-hot encoded 'embarked':")
print(pd.concat([df['embarked'], df_ohe], axis=1).head(10))

One-hot encoded 'embarked':
  embarked  embarked_Q  embarked_S
0        S       False        True
1        C       False       False
2        S       False        True
3        S       False        True
4        S       False        True
5        Q        True       False
6        S       False        True
7        S       False        True
8        S       False        True
9        C       False       False


In [6]:
# Full encoding of multiple columns at once
cols_to_encode = ['sex', 'embarked']
df_full = df[['survived', 'pclass', 'age', 'fare', 'sex', 'embarked']].dropna().copy()

df_encoded = pd.get_dummies(df_full, columns=cols_to_encode, drop_first=True)

print("Before encoding — columns:", df_full.columns.tolist())
print("After encoding  — columns:", df_encoded.columns.tolist())
print("\nFirst 3 rows:")
print(df_encoded.head(3))

Before encoding — columns: ['survived', 'pclass', 'age', 'fare', 'sex', 'embarked']
After encoding  — columns: ['survived', 'pclass', 'age', 'fare', 'sex_male', 'embarked_Q', 'embarked_S']

First 3 rows:
   survived  pclass   age     fare  sex_male  embarked_Q  embarked_S
0         0       3  22.0   7.2500      True       False        True
1         1       1  38.0  71.2833     False       False       False
2         1       3  26.0   7.9250     False       False        True


## Handling High-Cardinality Columns

One-hot encoding a column with hundreds of unique values creates hundreds of new columns — a problem called the *curse of dimensionality*.

**Strategies**:
1. Group rare categories into an "Other" bucket, then OHE
2. Use target encoding (encode by mean of the target within each category)
3. Use embeddings (for very high cardinality)

In [7]:
# Simulate high-cardinality column (embark_town has 3 values here, but imagine a city column)
town_counts = df['embark_town'].value_counts()
print("embark_town counts:\n", town_counts)

# Group rare categories (below a threshold) into 'Other'
threshold = 50
rare = town_counts[town_counts < threshold].index
df['embark_town_grouped'] = df['embark_town'].apply(
    lambda x: 'Other' if x in rare else x
)
print("\nAfter grouping rare categories:")
print(df['embark_town_grouped'].value_counts())

embark_town counts:
 embark_town
Southampton    644
Cherbourg      168
Queenstown      77
Name: count, dtype: int64

After grouping rare categories:
embark_town_grouped
Southampton    644
Cherbourg      168
Queenstown      77
Name: count, dtype: int64


## Decision Guide

```
Is the column ordinal (has a meaningful order)?
   YES → Label encoding with explicit mapping
   NO  → Is cardinality low (< ~15 categories)?
            YES → One-hot encoding
            NO  → Group rare categories first, then OHE
                  OR use target encoding
```

| Method | When | Pandas | sklearn |
|---|---|---|---|
| Label encoding | Ordinal | `.map(dict)` | `LabelEncoder` |
| One-hot encoding | Nominal | `pd.get_dummies()` | `OneHotEncoder` |